In [ ]:
# Bank Customer Churn Prediction using Neural Network

# 1. Import Libraries
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# 2. Load Dataset
df = pd.read_csv("Churn_Modelling.csv")

print(df.head())
print(df.shape)
print(df.columns)

# 3. Data Preprocessing
df = df.drop(
    columns=["RowNumber", "CustomerId", "Surname"],
    errors="ignore"
)

X = df.drop("Exited", axis=1)
y = df["Exited"]

# Convert categorical columns into numeric columns
X = pd.get_dummies(X, drop_first=True, dtype=int)

# 4. Split Dataset into Training and Testing Sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

# 5. Feature Scaling
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

print("Data normalization completed.")

# 6. Initialize Neural Network Model
model = Sequential([
    Dense(32, activation="relu", input_shape=(X_train.shape[1],)),
    Dropout(0.2),
    Dense(16, activation="relu"),
    Dense(1, activation="sigmoid")
])

# 7. Compile Neural Network
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# 8. Early Stopping
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

# 9. Train Neural Network
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

print("Neural network training completed.")

# 10. Make Predictions
y_prob = model.predict(X_test)

# Convert probabilities into class labels
y_pred = (y_prob >= 0.5).astype(int).flatten()

# 11. Calculate Accuracy
accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy Score:", accuracy)

# 12. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(cm)

# 13. Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))